<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">
</p>

# **NOTEBOOK 7 BIS**
---
# **Naive Bayes para clasificación de texto**

## Idea intuitiva

Naive Bayes es un clasificador probabilístico muy usado en NLP. La idea es sencilla: dado un documento, calculamos cuál de las clases posibles es más probable que lo haya generado.

En lugar de intentar modelar todo el texto como un fenómeno complejo, asumimos que las palabras son características independientes una vez conocida la clase. Esta suposición es simplificada, pero funciona muy bien en muchos problemas reales.

La regla que se usa es:

$$
P(C \mid D) = \frac{P(D \mid C) \cdot P(C)}{P(D)}
$$

Donde:
- $C$ es la clase (por ejemplo, Cine o Literatura).
- $D$ es el documento (el texto a clasificar).
- $P(C)$ es la probabilidad a priori de la clase.
- $P(D \mid C)$ es la probabilidad de observar ese documento si sabemos que pertenece a esa clase.

Como $P(D)$ es el mismo para todas las clases, lo que realmente queremos maximizar es:

$$
P(C) \cdot P(D \mid C)
$$

Y si el documento está representado por palabras $w_1, w_2, \dots, w_n$, entonces:

$$
P(D \mid C) = \prod_{i=1}^{n} P(w_i \mid C)
$$

Por tanto, la decisión final es:

$$
\hat{C} = \arg\max_C \left[ P(C) \prod_{i=1}^{n} P(w_i \mid C) \right]
$$

## ¿Por qué se llama “Naive”?

Porque asumimos que las palabras son independientes entre sí, dado que conocemos la clase. Eso no es estrictamente cierto en lenguaje natural, pero en la práctica es una hipótesis muy útil.

Por ejemplo, en un texto sobre cine, las palabras "película", "actores" y "guion" suelen aparecer juntas. Sin embargo, el modelo ignora esa correlación y considera cada una por separado. Aun así, el resultado suele ser sorprendentemente bueno.

La clave es que el clasificador no intenta entender el significado completo del texto, sino estimar qué clase es más probable que haya generado esas palabras.

## Ejemplo pequeño

Supongamos que queremos clasificar frases entre dos clases: `Cine` y `Literatura`.

Usamos este conjunto de entrenamiento:

1. "La película fue emocionante y llena de acción." → Cine
2. "Ese libro tiene una trama intrigante." → Literatura
3. "Los actores hicieron un trabajo excelente." → Cine
4. "El autor describe paisajes con gran detalle." → Literatura
5. "El cine de autor siempre me ha fascinado." → Cine
6. "La novela estaba llena de giros inesperados." → Literatura
7. "El guion de esa película fue escrito por un famoso novelista." → Cine
8. "Los personajes del libro eran muy realistas." → Literatura
9. "Esa película está basada en un libro aclamado." → Cine

Las probabilidades a priori son:

$$
P(Cine) = \frac{5}{9}, \quad P(Literatura) = \frac{4}{9}
$$

Es decir, antes de mirar el texto, una frase tiene más probabilidad de ser de cine que de literatura.

In [ ]:
import re
from collections import Counter

documents = [
    'La película fue emocionante y llena de acción.',
    'Ese libro tiene una trama intrigante.',
    'Los actores hicieron un trabajo excelente.',
    'El autor describe paisajes con gran detalle.',
    'El cine de autor siempre me ha fascinado.',
    'La novela estaba llena de giros inesperados.',
    'El guion de esa película fue escrito por un famoso novelista.',
    'Los personajes del libro eran muy realistas.',
    'Esa película está basada en un libro aclamado.'
]

labels = [0, 1, 0, 1, 0, 1, 0, 1, 0]  # 0 = Cine, 1 = Literatura
label_names = {0: 'Cine', 1: 'Literatura'}

# Preprocesado sencillo: minúsculas y eliminación de signos
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\sáéíóúñü]', ' ', text)
    return text.split()

docs = [preprocess(d) for d in documents]
print('Documentos preprocesados:')
for i, d in enumerate(docs, 1):
    print(f'{i}. {d}')

# Priors
prior = {cls: labels.count(cls) / len(labels) for cls in set(labels)}
print('\nPriors:')
for cls, p in prior.items():
    print(label_names[cls], p)

# Conteo de palabras por clase
counts = {cls: Counter() for cls in set(labels)}
for doc, cls in zip(docs, labels):
    counts[cls].update(doc)

print('\nConteos por clase:')
for cls in sorted(counts):
    print(label_names[cls], dict(counts[cls]))

# Número total de palabras por clase
total_words = {cls: sum(counts[cls].values()) for cls in counts}
print('\nTotal de palabras por clase:')
for cls in sorted(total_words):
    print(label_names[cls], total_words[cls])

## Cálculo de probabilidades condicionales

Para cada palabra se calcula:

$$
P(w \mid C) = \frac{\#(w, C)}{\#(C)}
$$

donde $\#(w, C)$ es el número de veces que aparece la palabra $w$ en la clase $C$ y $\#(C)$ es el número total de palabras de los documentos de esa clase.

Por ejemplo:

$$
P(\text{libro} \mid Literatura) = \frac{2}{15}
$$

y

$$
P(\text{libro} \mid Cine) = \frac{1}{21}
$$

Esto explica por qué la palabra "libro" influye más cuando la clase es Literatura que cuando es Cine.

In [ ]:
from math import log

# Vocabulario completo
vocab = sorted({word for doc in docs for word in doc})
print('Tamaño del vocabulario:', len(vocab))

# Probabilidades condicionales con suavizado de Laplace
alpha = 1.0
conditional = {}
for cls in set(labels):
    conditional[cls] = {}
    total = sum(counts[cls].values())
    for word in vocab:
        num = counts[cls].get(word, 0) + alpha
        den = total + alpha * len(vocab)
        conditional[cls][word] = num / den

print('\nProbabilidades condicionales ejemplo:')
for word in ['libro', 'autor', 'película', 'paisajes']:
    print(word, {label_names[cls]: round(conditional[cls].get(word, 0), 4) for cls in sorted(conditional)})

# Un ejemplo de predicción
new_doc = preprocess('El libro es un autor fascinante y una novela intrigante.')
print('\nDocumento a clasificar:', new_doc)

scores = {}
for cls in sorted(prior):
    score = log(prior[cls])
    for word in new_doc:
        score += log(conditional[cls].get(word, 1 / (sum(counts[cls].values()) + alpha * len(vocab))))
    scores[cls] = score
    print(label_names[cls], 'score =', score)

predicted = max(scores, key=scores.get)
print('\nPredicción:', label_names[predicted])

## ¿Qué está haciendo el algoritmo exactamente?

Para cada clase, Naive Bayes evalúa cuán probable es que ese documento haya sido generado por esa clase.

En texto, esto significa que calcula:

$$
score(C) = \log P(C) + \sum_{w \in D} \log P(w \mid C)
$$

y luego elige la clase con mayor valor.

Esto se hace en logaritmos porque multiplicar muchas probabilidades puede dar resultados muy pequeños y producir *underflow* numérico. El log no cambia el orden de las probabilidades, pero hace el cálculo más estable.

En otras palabras, Naive Bayes compara diferentes hipótesis, una por cada clase, y se queda con la más probable.

## Importancia del suavizado

Un problema típico es que una palabra puede no aparecer nunca en una clase dentro del conjunto de entrenamiento. Entonces:

$$
P(w \mid C) = 0
$$

y, al multiplicar todas las probabilidades, el resultado final puede volverse cero aunque la palabra no sea imposible fuera del conjunto de entrenamiento.

Para evitarlo se usa el suavizado de Laplace:

$$
P(w \mid C) = \frac{\#(w, C) + \alpha}{\#(C) + \alpha \cdot |V|}
$$

donde $|V|$ es el tamaño del vocabulario. Esto garantiza que ninguna palabra tenga probabilidad exactamente cero.

El valor más habitual es $\alpha = 1$.

Este detalle es importante porque hace que el algoritmo sea mucho más robusto en textos reales.

## Conclusión

Naive Bayes es un clasificador muy eficaz y muy útil en textos porque:

- usa probabilidades y la intuición de Bayes,
- es rápido y fácil de implementar,
- funciona bien con palabras y frecuencias,
- y permite manejar tareas como clasificación de spam, análisis de sentimiento o etiquetado temático.

La clave no está en que la independencia entre palabras sea real, sino en que el modelo usa esa simplificación para construir una decisión probabilística útil y eficiente.

---

### Ejercicio

Modifica el ejemplo para clasificar tres clases: `Cine`, `Literatura` y `Música`.

Puedes hacerlo ampliando el conjunto de entrenamiento y dejando la misma lógica de conteos y probabilidades.